# HouseholdBasketEnv — GRPO Training (Phases 6 + 7 + 8)

QLoRA + GRPO fine-tuning of **Qwen2.5-3B-Instruct** on `HouseholdBasketEnv`. This notebook runs:

- **MAIN** — full reward (default; plan §6)
- **Ablation A** — `enable_meal_type_coverage=False` (plan §7.A)
- **Ablation B** — train on Task 3 only, no curriculum mix (plan §7.B)

Switch experiments via the `RUN_NAME` and `ABLATION` cell at the top.

> **Hardware:** Google Colab T4 / A100. T4 fits 3B in 4-bit + LoRA-r32. Expect ~6–8h for the main run at 800 steps.
>
> **Seeds:** drawn from `seeds/verified_task{2,3}.json` (training_seeds split). Held-out seeds are NEVER used during training.

## 1. Run config (edit me)

Choose ONE: `"main"`, `"ablation_a"`, or `"ablation_b"`. Outputs land in `runs/<RUN_NAME>/`.

In [ ]:
# =============================================================================
# Pick exactly ONE run.
# =============================================================================
RUN_NAME = "main"            # "main" | "ablation_a" | "ablation_b"
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

# Trainer hyperparameters (plan §6.4) ----------------------------------------
MAX_TRAIN_STEPS = 800
PER_DEVICE_BATCH = 1
GRAD_ACCUM = 8
NUM_GENERATIONS = 8       # GRPO group size
GRPO_BETA = 0.04          # KL penalty
LEARNING_RATE = 5e-6
TEMPERATURE = 1.0
MAX_NEW_TOKENS = 96
SAVE_EVERY = 200
SEED = 0

# LoRA / 4-bit ---------------------------------------------------------------
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
LOAD_IN_4BIT = True
MAX_SEQ_LEN = 2048

# Curriculum mix (plan §6.2) -------------------------------------------------
TIER_WEIGHTS = {2: 0.7, 3: 0.3}   # default: main + ablation_a
if RUN_NAME == "ablation_b":
    TIER_WEIGHTS = {3: 1.0}

# Reward ablation (plan §7.A) ------------------------------------------------
ENABLE_MEAL_TYPE_COVERAGE = (RUN_NAME != "ablation_a")

print({
    "RUN_NAME": RUN_NAME,
    "TIER_WEIGHTS": TIER_WEIGHTS,
    "ENABLE_MEAL_TYPE_COVERAGE": ENABLE_MEAL_TYPE_COVERAGE,
    "MAX_TRAIN_STEPS": MAX_TRAIN_STEPS,
})

## 2. Bootstrap (Colab + paths)

In [ ]:
# --- Colab / local bootstrap ------------------------------------------------
# Colab : git clone the standalone repo into /content/HouseholdBasketEnv
# Local : auto-detect the household_basket_env package by walking up from cwd
import os, sys, pathlib, subprocess

ON_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/I-am-vishalmaurya/HouseholdBasketEnv"

def _find_env_root(start: pathlib.Path) -> pathlib.Path:
    for p in [start, *start.parents]:
        if (p / "household_basket_env" / "server" / "environment.py").exists():
            return p
    raise RuntimeError(f"Could not locate household_basket_env package starting from {start}")

if ON_COLAB:
    REPO_ROOT = pathlib.Path("/content/HouseholdBasketEnv")
    if not REPO_ROOT.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)])
    else:
        print(f"[bootstrap] {REPO_ROOT} already exists; skipping clone.")
else:
    REPO_ROOT = _find_env_root(pathlib.Path.cwd())

PKG_ROOT = REPO_ROOT
ENV_ROOT = REPO_ROOT / "household_basket_env"
if str(PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(PKG_ROOT))
os.environ["HOUSEHOLD_BASKET_PRODUCTS_PATH"] = str(ENV_ROOT / "data" / "products.json")
os.environ.setdefault("HF_HOME", str(REPO_ROOT / ".hf_cache"))

RUN_DIR = ENV_ROOT / "runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("REPO_ROOT =", REPO_ROOT)
print("RUN_DIR   =", RUN_DIR)

In [ ]:
if ON_COLAB:
    !pip -q install --upgrade pip
    !pip -q install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    !pip -q install --no-deps "trl<0.20.0" peft accelerate bitsandbytes
    !pip -q install pydantic==2.* fastapi uvicorn

# Sanity import the env package -----------------------------------------------
from household_basket_env.server.environment import HouseholdBasketEnvironment
from household_basket_env.models import BasketAction
from household_basket_env.server.curriculum import (
    TIER_CONFIGS,
    load_verified_seeds,
    TRAIN_TIER_WEIGHTS,
)
print("Env package imports OK.")

## 3. Load Qwen2.5-3B-Instruct + LoRA

In [ ]:
from unsloth import FastLanguageModel, PatchFastRL
PatchFastRL("GRPO", FastLanguageModel)  # tells Unsloth to patch GRPO before model load

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=LOAD_IN_4BIT,
    fast_inference=True,            # vLLM-style sampling speedup
    max_lora_rank=LORA_R,
    gpu_memory_utilization=0.55,    # safe T4 default
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
print("Model + LoRA ready.")

## 4. Reward function

GRPO calls `reward_fn(prompts, completions)` once per group of `NUM_GENERATIONS`. For HouseholdBasketEnv we don't have a one-shot reward — we have an **episode** of up to `max_steps` actions. Strategy:

1. Each prompt encodes the **current observation only** for one specific (seed, step) tuple.
2. We use the model's completion as the action, step the env, and return the per-step reward.
3. GRPO group sampling explores diverse completions per state — exactly what we want for credit assignment in a sparse-terminal env.

This is a **per-step bandit** view of GRPO (matches Unsloth's GRPOTrainer + open-source recipes). For full episode-level credit, you can swap to TRL's PPOTrainer or a custom rollout loop; the per-step variant is sufficient for our reward shaping.

In [ ]:
import json, re, random
from textwrap import dedent

SYSTEM_PROMPT = """You are a careful nutrition-aware shopping assistant for an Indian household. \
At each step you pick exactly ONE product from the catalog and assign it to ONE household member. \
You MUST output ONLY a JSON object with three keys: \"product_id\" (string), \"member_id\" (string), \"rationale\" (1 short sentence). \
Do NOT add any prose, code fences, or extra keys. Respect each member's caps and allergies; spread meal types across breakfast/lunch/dinner/snack/beverage; stay under the budget."""

JSON_RE = re.compile(r"\{[^{}]*\}", re.S)

def render_observation(obs) -> str:
    members = []
    for m in obs.household:
        caps = ", ".join(f"{k}={v:.0f}" for k, v in m.thresholds_cap.items())
        members.append(f"- {m.member_id} | conds={m.health_conditions} | restr={m.dietary_restrictions} | caps: {caps}")
    cands = []
    for c in obs.candidates[:50]:
        cands.append(
            f"- {c.product_id} | {c.category} | meal={c.meal_type} | INR {c.price_inr:.0f} | "
            f"sugars_g={c.nutrients.get('sugars_g',0):.1f} sodium_mg={c.nutrients.get('sodium_mg',0):.0f} fat_g={c.nutrients.get('fat_g',0):.1f}"
        )
    basket = [f"- {t.product_id} -> {t.member_id}" for t in obs.basket_so_far]
    return dedent(f"""
    Household:
    {chr(10).join(members)}

    Catalog (top 50):
    {chr(10).join(cands)}

    Basket so far ({len(basket)}):
    {chr(10).join(basket) if basket else "(empty)"}

    Budget remaining: INR {obs.budget_remaining:.0f}
    Steps used: {obs.step_index} / {obs.max_steps}
    Attempts used: {obs.attempt_index} / {obs.max_steps * 4}

    Now pick ONE product and assign it to ONE member. Output JSON only.
    """).strip()

def extract_json(text: str):
    m = JSON_RE.search(text)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

print("Prompt utilities ready.")

In [ ]:
# Build per-step training prompts from the verified seed pool ----------------
import itertools

train_pool = []  # list of (seed, task_id)
for tier_id, w in TIER_WEIGHTS.items():
    payload = load_verified_seeds(tier_id)
    for s in payload["training_seeds"]:
        train_pool.append((s, tier_id))

if not train_pool:
    raise RuntimeError("Empty training pool. Re-run seed_verifier.")

print(f"Training pool: {len(train_pool)} (seed, tier) pairs across tiers {sorted(TIER_WEIGHTS.keys())}.")

# A persistent env that we reset per-step ------------------------------------
TRAIN_ENV = HouseholdBasketEnvironment()

def sample_starting_observation(rng: random.Random):
    """Pick a (seed, tier) by tier weight, reset env, return rendered prompt + obs ref."""
    tiers = list(TIER_WEIGHTS.keys())
    weights = [TIER_WEIGHTS[t] for t in tiers]
    tier_id = rng.choices(tiers, weights=weights, k=1)[0]
    payload = load_verified_seeds(tier_id)
    seed = rng.choice(payload["training_seeds"])
    obs = TRAIN_ENV.reset(seed=seed, task_id=tier_id)
    return seed, tier_id, obs

# Build a prompt set for the trainer. GRPOTrainer expects a HF Dataset of prompts.
import datasets

def make_prompt_record(rng: random.Random):
    seed, tier_id, obs = sample_starting_observation(rng)
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": render_observation(obs)},
        ],
        "seed": seed,
        "task_id": tier_id,
    }

# Generate 4096 prompt seeds; the trainer iterates with replacement.
PROMPT_RNG = random.Random(SEED)
prompt_rows = [make_prompt_record(PROMPT_RNG) for _ in range(4096)]
prompt_ds = datasets.Dataset.from_list(prompt_rows)
print("Prompt dataset:", prompt_ds)

In [ ]:
# --- Reward function (per-step bandit view of GRPO) -------------------------
# We replay the env to the prompt's starting state, then score the completion.

from household_basket_env.server.environment import HouseholdBasketEnvironment

# Use a thread-local env so concurrent reward evaluation doesn't cross-talk.
def reward_basket_step(prompts, completions, **kwargs):
    """Returns one float per (prompt, completion). Plan §4 dense rewards.

    The trainer passes `seed` and `task_id` columns through `**kwargs` thanks to
    GRPOTrainer's `remove_unused_columns=False` flag below.
    """
    seeds = kwargs["seed"]
    task_ids = kwargs["task_id"]
    rewards = []
    env = HouseholdBasketEnvironment(enable_meal_type_coverage=ENABLE_MEAL_TYPE_COVERAGE)
    for completion, seed, task_id in zip(completions, seeds, task_ids):
        env.reset(seed=int(seed), task_id=int(task_id))
        text = completion[0]["content"] if isinstance(completion, list) else completion
        payload = extract_json(text)
        if payload is None:
            payload = text  # P_parse path
        out = env.apply_raw_action(payload)
        rewards.append(float(out.reward or 0.0))
    return rewards

# Fall back: the env may not yet support the kwarg. If so, monkey-patch.
import inspect
sig = inspect.signature(HouseholdBasketEnvironment.__init__)
if "enable_meal_type_coverage" not in sig.parameters:
    print("[warn] enable_meal_type_coverage kwarg not in env; ablation_a will silently no-op.")
print("Reward fn ready.")

## 5. GRPOTrainer

In [ ]:
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir=str(RUN_DIR),
    learning_rate=LEARNING_RATE,
    beta=GRPO_BETA,
    num_generations=NUM_GENERATIONS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    max_prompt_length=MAX_SEQ_LEN - MAX_NEW_TOKENS,
    max_completion_length=MAX_NEW_TOKENS,
    max_steps=MAX_TRAIN_STEPS,
    save_steps=SAVE_EVERY,
    logging_steps=10,
    seed=SEED,
    bf16=False,
    fp16=True,
    optim="adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    temperature=TEMPERATURE,
    remove_unused_columns=False,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_basket_step],
    args=config,
    train_dataset=prompt_ds,
)
print("Trainer constructed.")

In [ ]:
history = trainer.train()
print("Training finished.")

## 6. Save checkpoints + training log

In [ ]:
import json

ADAPTER_DIR = RUN_DIR / "adapter_final"
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print("Saved final adapters ->", ADAPTER_DIR)

log_path = RUN_DIR / "training_log.json"
with open(log_path, "w", encoding="utf-8") as f:
    json.dump(trainer.state.log_history, f, indent=2)
print("Saved log ->", log_path)

config_path = RUN_DIR / "run_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump({
        "RUN_NAME": RUN_NAME,
        "MODEL_NAME": MODEL_NAME,
        "TIER_WEIGHTS": TIER_WEIGHTS,
        "ENABLE_MEAL_TYPE_COVERAGE": ENABLE_MEAL_TYPE_COVERAGE,
        "MAX_TRAIN_STEPS": MAX_TRAIN_STEPS,
        "PER_DEVICE_BATCH": PER_DEVICE_BATCH,
        "GRAD_ACCUM": GRAD_ACCUM,
        "NUM_GENERATIONS": NUM_GENERATIONS,
        "GRPO_BETA": GRPO_BETA,
        "LEARNING_RATE": LEARNING_RATE,
        "TEMPERATURE": TEMPERATURE,
        "MAX_NEW_TOKENS": MAX_NEW_TOKENS,
        "LORA_R": LORA_R,
        "LORA_ALPHA": LORA_ALPHA,
        "LORA_DROPOUT": LORA_DROPOUT,
        "LOAD_IN_4BIT": LOAD_IN_4BIT,
    }, f, indent=2)
print("Saved config ->", config_path)

### Next steps

- For each ablation, restart the notebook with a different `RUN_NAME` cell value.
- Run `eval_and_plots.ipynb` to compare against `docs/results/baseline.json`.
- Optional: push `adapter_final/` to HuggingFace via `scripts/push_to_hf.sh`.